In [ ]:
import json
import pandas as pd

from pathlib import Path

In [ ]:
UNKNOWN_CLASS_LABEL = "unk"

# ISIC 2020

In [ ]:
ISIC2020_DATASET_PATH = Path("/home/sulcm/datasets/isic2020/isic2020_builder/train")

In [ ]:
isic2020_gt = pd.read_csv(ISIC2020_DATASET_PATH / "ISIC_2020_Training_GroundTruth.csv")

In [ ]:
isic2020_gt["diagnosis"].value_counts()

# ISIC 2019

In [ ]:
ISIC2019_DATASET_PATH = Path("/home/sulcm/datasets/isic2019")
COMMON_LABEL_MAPPING = { # isic2019 -> common labels
    "nv": "nv",
    "mel": "mel",
    "bkl": "bkl",
    "df": "df",
    "scc": "sccka",
    "bcc": "bcc",
    "vasc": "vasc",
    "ak": "akiec",
}

### Train

In [ ]:
isic2019_gt_one_hot = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Training_GroundTruth.csv").set_index("image", append=True)
for column in isic2019_gt_one_hot.columns:
    isic2019_gt_one_hot[column] = isic2019_gt_one_hot[column].astype(int)
labels = [l.lower() for l in isic2019_gt_one_hot.columns.to_list()]
isic2019_gt_classes = isic2019_gt_one_hot.dot(isic2019_gt_one_hot.columns).apply(lambda x: x.lower())

In [ ]:
isic2019_training_input = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Training_Metadata.csv")
isic2019_training_input

In [ ]:
isic2019_training_input[["file_name", "label"]] = isic2019_training_input.apply(
    lambda row: [
        row["image"] + ".jpg",
        COMMON_LABEL_MAPPING[isic2019_gt_classes.xs(row["image"], level=1).iloc[0]]
    ],
    axis=1, result_type="expand"
)

ds_info = {
    "labels": isic2019_training_input["label"].unique().tolist()
}

In [ ]:
isic2019_training_input.to_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "train" / "metadata.csv", index=False)
with open(ISIC2019_DATASET_PATH / "isic2019_builder" / "dataset_info.json", "w") as f:
    json.dump(ds_info, f, indent=2, ensure_ascii=False)

### Validation

In [ ]:
isic2019_val_gt_one_hot = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Test_GroundTruth.csv").set_index("image", append=True)
isic2019_val_gt_one_hot.drop(columns=["score_weight", "validation_weight"], inplace=True)
for column in isic2019_val_gt_one_hot.columns:
    isic2019_val_gt_one_hot[column] = isic2019_val_gt_one_hot[column].astype(int)
labels = [l.lower() for l in isic2019_val_gt_one_hot.columns.to_list()]
isic2019_gt_classes = isic2019_val_gt_one_hot.dot(isic2019_val_gt_one_hot.columns).apply(lambda x: x.lower())

In [ ]:
isic2019_val_input = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Test_Metadata.csv")
isic2019_val_input

In [ ]:
isic2019_val_input[["file_name", "label", "lesion_id"]] = isic2019_val_input.apply(
    lambda row: [
        row["image"] + ".jpg",
        COMMON_LABEL_MAPPING.get(isic2019_gt_classes.xs(row["image"], level=1).iloc[0], UNKNOWN_CLASS_LABEL),
        "UNK_" + row["image"]
    ],
    axis=1, result_type="expand"
)


len_with_unk_labels = len(isic2019_val_input)
isic2019_val_input = isic2019_val_input.loc[isic2019_val_input["label"] != UNKNOWN_CLASS_LABEL]
print(f"Valid labels for {len(isic2019_val_input)}/{len_with_unk_labels}")
isic2019_val_input

In [ ]:
isic2019_val_input.to_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "validation" / "metadata.csv", index=False)

In [ ]:
from datasets import load_dataset, load_from_disk, ClassLabel

In [ ]:
dataset = load_dataset("imagefolder", data_dir=ISIC2019_DATASET_PATH / "isic2019_builder")
dataset

In [ ]:
dataset = dataset.cast_column("label", ClassLabel(names=list(COMMON_LABEL_MAPPING.values())))

In [ ]:
dataset.save_to_disk(ISIC2019_DATASET_PATH / "isic2019")

In [ ]:
lds = load_from_disk(
    dataset_path=ISIC2019_DATASET_PATH / "isic2019"
)

In [ ]:
lds["train"].features